### **Import libraries**: 
- **Defeat Beta API** is an open-source alternative to Yahoo Finance's market data APIs. Moreover, it has the added ability of permitting news retrieval.
- **Hugging Face** is a platform and library for utilizing machine learning models.
- **InferenceClient** is a library available on Hugging Face and is primarily used to perform inference – the process of reaching conclusions given a set of inputs.
- **Transformers** is a Python library created by Hugging Face that allows one to download, manipulate, and run thousands of pretrained, open-source AI models.
- **Pipeline** is a package within Transformers that simplifies the process of working with advanced AI models.
- **UUID** is a library that provides a way to generate universally unique identifiers. In this script, it is used to create identifiers for headlines.

In [1]:
import requests
import defeatbeta_api
from defeatbeta_api.data.ticker import Ticker
from defeatbeta_api.data.tickers import Tickers
from huggingface_hub import InferenceClient
from transformers import pipeline
import uuid
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import sys
import fastparquet
import os
from IPython.display import clear_output
import time

[nltk_data] Downloading package punkt_tab to /tmp/defeatbeta/nltk...
[nltk_data]   Package punkt_tab is already up-to-date!


______      __           _    ______      _        
|  _  \    / _|         | |   | ___ \    | |       
| | | |___| |_ ___  __ _| |_  | |_/ / ___| |_ __ _ 
| | | / _ \  _/ _ \/ _` | __| | ___ \/ _ \ __/ _` |
| |/ /  __/ ||  __/ (_| | |_  | |_/ /  __/ || (_| |
|___/ \___|_| \___|\__,_|\__| \____/ \___|\__\__,_|
📈:: Data Update Time ::	2026-04-03 ::
📈:: Software Version ::	0.0.50      ::


### **Create a data frame with ticker, headlineID, headline, article publication date, and article source columns.** ###

In [2]:
def package(ticker_lst_param):
    df_main = pd.DataFrame(columns = ["Ticker", "HeadlineID", "Headline", "Timestamp", "Source"])
    for element in ticker_lst_param:
        tickers = [element["Ticker"] for i in range(len(element["Headline"]))]
        headline_IDs = [str(uuid.uuid4())[:8] for i in range(len(element["Headline"]))]
        data = {"Ticker" : tickers, "HeadlineID" : headline_IDs, "Headline" : element["Headline"], "Timestamp" : element["Timestamp"], "Source" : element["Source"]}
        df = pd.DataFrame(data = data)
        df_main = pd.concat([df_main, df])
    return df_main

### **Use the Defeat Beta API to pull news associated with each ticker.**

### **To modify the tickers used, make changes to tickers_file.txt**

In [3]:
def get_data(tickers_file):
    with open(tickers_file, "r") as f:
        tickers = f.readline()
        tickers = tickers[1:-1]
        tickers = tickers.replace('"', "")
        tickers = tickers.split(",")
        tickers = [element.strip() for element in tickers]
        tickers_cl = Tickers(tickers)
    news = tickers_cl.news()
    headline_data = [{"Ticker" : element, "Headline" : news[element].get_news_list()["title"], "Timestamp" : news[element].get_news_list()["report_date"], "Source" : news[element].get_news_list()["publisher"]} for element in tickers]
    headline_data_packaged = package(headline_data)
    return headline_data_packaged, tickers

### **Utilize FinBERT - a pre-trained NLP model specialized for financial sentiment analysis – adapted by ProsusAI to analyze the probabilities (positive/negative/neutral) associated with each headline.**

In [4]:
def add_probabilities(ticker_news_data_df_param, api_key_param):
    pipe = pipeline("text-classification", model="ProsusAI/finbert")
    client = InferenceClient(provider="hf-inference", api_key=api_key_param)
    ticker_news_data_df_param[["pos", "neu", "neg"]] = 0.0, 0.0, 0.0
    ticker_news_data_df_param["Timestamp"] = pd.to_datetime(ticker_news_data_df_param["Timestamp"])
    ticker_news_data_df_param = ticker_news_data_df_param[ticker_news_data_df_param["Timestamp"] >= (datetime.today() - timedelta(days=7))]
    rows_num, row_counter = len(ticker_news_data_df_param), 1
    for index, row in ticker_news_data_df_param.iterrows():
        headline = row["Headline"]
        result = client.text_classification(headline, model="ProsusAI/finbert")
        for probability in result:
            row = row.copy()
            match probability["label"]:
                case "positive":
                    ticker_news_data_df_param.loc[index, "pos"] = probability.score
                case "neutral":
                    ticker_news_data_df_param.loc[index, "neu"] = probability.score
                case "negative":
                    ticker_news_data_df_param.loc[index, "neg"] = probability.score
        
        clear_output()
        print(str(round((row_counter/rows_num)*100, 2)) + "% complete.")
        row_counter +=1
        time.sleep(0.2)

    ticker_news_data_df_param.set_index(["Ticker", "Timestamp", "HeadlineID"], inplace = True)
    ticker_news_data_df_param.sort_index(inplace = True, level = 0)
    ticker_news_data_df_param.sort_index(inplace = True, level = 1, ascending = False, sort_remaining = False)
    return ticker_news_data_df_param

### **Main program used to collectively execute the whole script and retrieve the API key used for the inference package.**

In [5]:
def main():
    startTime = datetime.now()
    with open("inference_api_key.txt", "r") as f:
        api_key = f.readline().strip()
    ticker_news_data_df, tickers = get_data("tickers.txt")
    ticker_news_data_df_with_probabilities = add_probabilities(ticker_news_data_df, api_key)
    #print(ticker_news_data_df_with_probabilities)
    #ticker_news_data_df_with_probabilities.to_parquet("daily_ticker_headlines_with_FinBERT_scoring_2.parquet")
    print(datetime.now() - startTime)
main()

0.08% complete.
2026-04-07 08:07:50 INFO httpx MainThread - HTTP Request: POST https://router.huggingface.co/hf-inference/models/ProsusAI/finbert "HTTP/1.1 504 Gateway Time-out"


HfHubHTTPError: Server error '504 Gateway Time-out' for url 'https://router.huggingface.co/hf-inference/models/ProsusAI/finbert' (Amz CF ID: w5HMCnZOG6hBgsQQ36_xSel4iaIZhg7wr1gCG0A5pY-TVhpa06FiDw==)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/504